# Exploratory data analysis: PHEME and MultiNLI

Dataset statistics, class balance and sanity checks for the Cross-Account Inconsistency Detector.
Everything is read from the local copies under `data/raw/`; nothing is downloaded.

Run from the repository root or from `notebooks/` (the first cell finds the root).

In [ ]:
import sys, json, collections
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from evaluation import plots                      # shared figure style and colours
%matplotlib inline
from pipeline.config import load_config, resolve_path
from data.pheme import load_pheme_event, list_events, convert_veracity, event_dir
from data.multinli import load_multinli
from data.proxy_labels import build_proxy_pairs

cfg = load_config()
PHEME = resolve_path(cfg, "pheme_v2")
MAIN = cfg["events"]["main"]
pd.set_option("display.max_colwidth", 110)
print("events on disk:", list_events(PHEME))

## 1. PHEME: what is on disk

Threads, replies and annotation keys for every event (raw files, before any cleaning).

In [ ]:
rows, ann_keys, nonrumour_keys = [], collections.Counter(), collections.Counter()
for event in list_events(PHEME):
    folder = event_dir(PHEME, event)
    row = {"event": event, "main": event in MAIN}
    for kind in ("rumours", "non-rumours"):
        threads = [t for t in (folder / kind).iterdir() if t.is_dir() and not t.name.startswith(".")] if (folder / kind).is_dir() else []
        replies = 0
        for t in threads:
            ann = json.loads((t / "annotation.json").read_text(encoding="utf-8"))
            (ann_keys if kind == "rumours" else nonrumour_keys).update(ann.keys())
            if (t / "reactions").is_dir():
                replies += sum(1 for p in (t / "reactions").iterdir() if p.suffix == ".json" and not p.name.startswith("."))
        row[f"{kind} threads"] = len(threads)
        row[f"{kind} replies"] = replies
    rows.append(row)
raw_stats = pd.DataFrame(rows).set_index("event")
raw_stats["rumour share"] = (raw_stats["rumours threads"] / (raw_stats["rumours threads"] + raw_stats["non-rumours threads"])).round(3)
raw_stats

In [ ]:
print("annotation.json keys in rumour threads    :", dict(ann_keys))
print("annotation.json keys in non-rumour threads:", dict(nonrumour_keys))
stance_like = [k for k in list(ann_keys) + list(nonrumour_keys) if any(s in k.lower() for s in ("stance", "support", "deny", "respon"))]
print("stance-like keys:", stance_like or "NONE -> PHEME here has no reply stance labels")

**Finding.** Rumour threads carry `category` (the rumour story), `misinformation`, `true`, `links` and `is_turnaround`.
Non-rumour threads carry only `is_rumour`. There are no stance labels and no entailment/contradiction labels,
so evaluation has to use veracity- and story-based proxies (see `data/proxy_labels.py`).

## 2. Claims after cleaning (main events, source tweets)

In [ ]:
claims, funnel = {}, []
for event in MAIN:
    raw = load_pheme_event(PHEME, event, languages=None, min_tokens=0, dedup=False)
    lang = load_pheme_event(PHEME, event, languages=["en"], min_tokens=0, dedup=False)
    short = load_pheme_event(PHEME, event, languages=["en"], min_tokens=cfg["data"]["min_tokens"], dedup=False)
    claims[event] = load_pheme_event(PHEME, event, languages=["en"], min_tokens=cfg["data"]["min_tokens"], dedup=True)
    funnel.append({"event": event, "source tweets": len(raw), "after language filter": len(lang),
                   "after min-token filter": len(short), "after dedup": len(claims[event]),
                   "accounts": len({c.account_id for c in claims[event]}),
                   "rumour claims": sum(c.metadata["is_rumour"] for c in claims[event])})
funnel = pd.DataFrame(funnel).set_index("event"); funnel

In [ ]:
for event in MAIN[:2]:
    c = claims[event][5]
    print(f"[{event}] raw    : {c.metadata['raw_text']!r}\n{' ' * (len(event) + 3)}cleaned: {c.text!r}\n")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.arange(0, 36, 2)
for colour, event in zip(plots.SERIES, MAIN):
    lengths = [len(c.text.split()) for c in claims[event]]
    ax.hist(lengths, bins=bins, histtype="step", lw=2, color=colour, density=True, label=f"{event} (median {int(np.median(lengths))})")
ax.set_xlabel("tokens per cleaned claim"); ax.set_ylabel("density"); ax.set_title("Claim length"); ax.legend(fontsize=8)
fig.savefig(ROOT / "outputs" / "figures" / "eda_claim_length.png"); plt.show()

## 3. Class balance: rumour veracity

In [ ]:
ver = pd.DataFrame({event: collections.Counter(c.proxy_label or "non-rumour" for c in claims[event]) for event in MAIN}).T.fillna(0).astype(int)
ver = ver[["true", "false", "unverified", "non-rumour"]]
display(ver)
display((ver.div(ver.sum(axis=1), axis=0) * 100).round(1).add_suffix(" %"))
fig, ax = plt.subplots(figsize=(8, 3.4))
left = np.zeros(len(ver))
for colour, col in zip([plots.SERIES[2], plots.SERIES[7], plots.SERIES[3], "#c3c2b7"], ver.columns):
    ax.barh(ver.index, ver[col], left=left, color=colour, edgecolor=plots.SURFACE, linewidth=1.5, label=col)
    left += ver[col].values
ax.set_xlabel("claims (deduplicated source tweets)"); ax.set_title("Veracity of the rumour story each claim reports")
ax.grid(axis="y", visible=False); ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.42), fontsize=8)
fig.savefig(ROOT / "outputs" / "figures" / "eda_veracity_balance.png"); plt.show()

## 4. Rumour stories (`category`): the weak reference for sub-event clustering

In [ ]:
story_rows, mixed = [], 0
for event in MAIN:
    by_story = collections.defaultdict(list)
    for c in claims[event]:
        if c.metadata["rumour_category"]:
            by_story[c.metadata["rumour_category"]].append(c)
    sizes = sorted((len(v) for v in by_story.values()), reverse=True)
    mixed += sum(len({c.proxy_label for c in v}) > 1 for v in by_story.values())
    story_rows.append({"event": event, "stories": len(by_story), "largest": sizes[0], "median size": int(np.median(sizes)),
                       "singleton stories": sum(s == 1 for s in sizes),
                       "true stories": sum(v[0].proxy_label == "true" for v in by_story.values()),
                       "false stories": sum(v[0].proxy_label == "false" for v in by_story.values()),
                       "unverified stories": sum(v[0].proxy_label == "unverified" for v in by_story.values()),
                       "turnaround tweets": sum(c.metadata["is_turnaround"] for v in by_story.values() for c in v),
                       "false-story turnarounds (debunks)": sum(c.metadata["is_turnaround"] and c.proxy_label == "false" for v in by_story.values() for c in v)})
print("stories whose threads mix veracity labels:", mixed)
pd.DataFrame(story_rows).set_index("event")

In [ ]:
event = "germanwings-crash"
top = collections.Counter((c.metadata["rumour_category"], c.proxy_label) for c in claims[event] if c.metadata["rumour_category"]).most_common(12)
pd.DataFrame([{"story": s, "veracity": v, "claims": n} for (s, v), n in top])

Veracity never varies inside a story, so "a cluster that mixes a true and a false story" and
"a debunking (turnaround) tweet next to the rumour it debunks" are the two disagreement signals the
annotations can give. Story sizes are very skewed (many singletons), which limits ARI/NMI as a clustering check.

## 5. Replies: language mix (why non-English text is filtered)

In [ ]:
lang_rows = {}
for event in MAIN:
    with_replies = load_pheme_event(PHEME, event, include_replies=True, languages=None, min_tokens=0, dedup=False)
    replies = [c for c in with_replies if not c.metadata["is_source"]]
    counts = collections.Counter(c.metadata["lang"] for c in replies)
    lang_rows[event] = {"replies": len(replies), **{l: counts.get(l, 0) for l in ("en", "und", "fr", "es", "ar", "de")},
                        "non-en %": round(100 * (1 - counts.get("en", 0) / max(1, len(replies))), 1)}
pd.DataFrame(lang_rows).T

## 6. Proxy pair labels: class balance

In [ ]:
rows = {}
for event in MAIN:
    pairs = build_proxy_pairs([c for c in claims[event] if c.metadata["is_rumour"]], per_class=cfg["evaluation"]["proxy_pairs_per_class"], seed=cfg["seed"])
    rows[event] = collections.Counter(label for _, _, label in pairs)
proxy = pd.DataFrame(rows).T.fillna(0).astype(int)[["entailment", "neutral", "contradiction"]]; proxy

In [ ]:
a, b, _ = next(p for p in build_proxy_pairs([c for c in claims["germanwings-crash"] if c.metadata["is_rumour"]], 400, cfg["seed"]) if p[2] == "contradiction")
print("contradiction-like proxy pair (debunk vs assertion of a false story):\n  A:", a.text, "\n  B:", b.text)

## 7. MultiNLI

In [ ]:
MNLI = resolve_path(cfg, "multinli")
rows, genres = {}, {}
for split in ("dev_matched", "dev_mismatched"):
    ex = load_multinli(MNLI, split)
    rows[split] = {"examples": len(ex), **collections.Counter(e.label for e in ex),
                   "median premise tokens": int(np.median([len(e.premise.split()) for e in ex])),
                   "median hypothesis tokens": int(np.median([len(e.hypothesis.split()) for e in ex]))}
    genres[split] = dict(collections.Counter(e.genre for e in ex))
n_train = sum(1 for _ in open(MNLI / "multinli_1.0_train.jsonl", encoding="utf-8"))
print("train examples (not used: the scorer is pretrained, fine-tuning is off by default):", n_train)
display(pd.DataFrame(rows).T); pd.DataFrame(genres).fillna(0).astype(int)

## 8. Sanity checks

In [ ]:
for event in MAIN:
    cs = claims[event]
    assert len({c.claim_id for c in cs}) == len(cs), "claim ids must be unique"
    assert all(c.text and "http" not in c.text and "@" not in c.text for c in cs), "cleaning left URLs or mentions"
    assert all(c.metadata["lang"] == "en" for c in cs)
    assert all((c.proxy_label is None) == (not c.metadata["is_rumour"]) for c in cs), "every rumour claim has a veracity"
    assert all(c.timestamp for c in cs)
assert mixed == 0
assert all(abs(rows[s]["entailment"] - rows[s]["contradiction"]) / rows[s]["examples"] < 0.05 for s in rows), "MultiNLI dev is balanced"
print("all sanity checks passed")